In [ ]:
!pip install -q transformers datasets peft bitsandbytes trl accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B"

# 1. Define the 4-bit quantization rules
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # float16 works best on Colab T4s
)

# 2. Load the Tokenizer and the squeezed Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Fix a common tokenizer quirk for training
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from peft import LoraConfig,get_peft_model


lora_config=LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model=get_peft_model(model,lora_config)

model.print_trainable_parameters()


In [ ]:
from datasets import Dataset

# 5 examples of sloppy human text -> perfect JSON
raw_data = [
    {"input": "Bought two packs of gum at 7-Eleven for 1.99 each", "output": '{"store": "7-Eleven", "items": [{"name": "gum", "qty": 2, "unit_price": 1.99}]}'},
    {"input": "Paid $45.50 for an oil change at Jiffy Lube on Tuesday", "output": '{"store": "Jiffy Lube", "items": [{"name": "oil change", "qty": 1, "unit_price": 45.50}]}'},
    {"input": "Grabbed 3 coffees from Dunkin, total came out to 12.45", "output": '{"store": "Dunkin", "items": [{"name": "coffee", "qty": 3, "unit_price": 4.15}]}'},
    {"input": "Target run: diapers were 24.99 and wipes were 4.50.", "output": '{"store": "Target", "items": [{"name": "diapers", "qty": 1, "unit_price": 24.99}, {"name": "wipes", "qty": 1, "unit_price": 4.50}]}'},
    {"input": "Got a single taco at Taco Bell for 2.10", "output": '{"store": "Taco Bell", "items": [{"name": "taco", "qty": 1, "unit_price": 2.10}]}'}
]

# THE FIX: Notice the {tokenizer.eos_token} appended to the very end of the output.
# This explicitly teaches the model that generating the EOS token is the final step!
def format_prompt(row):
    prompt = f"Extract transaction details to JSON.\nInput: {row['input']}\nOutput: {row['output']}{tokenizer.eos_token}"
    return {"text": prompt}

dataset = Dataset.from_list(raw_data).map(format_prompt)

In [ ]:
from trl import SFTTrainer, SFTConfig

# In the latest versions of the `trl` library, layout parameters
# like the text field and max length are moved into SFTConfig
training_args = SFTConfig(
    output_dir="./lora_cashier",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=20,
    learning_rate=2e-4,
    logging_steps=5,
    optim="paged_adamw_8bit",
    save_strategy="no",

    # --- NEW PLACEMENT FOR THESE ARGUMENTS ---
    dataset_text_field="text",
    max_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,  # 'tokenizer' argument was renamed to 'processing_class'
)

# Kick off the training!
trainer.train()

In [ ]:
# A totally unseen test prompt
test_input = "Extract transaction details to JSON.\nInput: Snagged 4 energy drinks at CVS for 11.20 total.\nOutput:"

inputs = tokenizer(test_input, return_tensors="pt").to("cuda")

# THE FIX: Switch the model to Evaluation mode to stop the gradient caching warnings
model.eval()

# THE FIX: Add explicit generation boundaries (temperature and eos_token_id)
outputs = model.generate(
    **inputs,
    max_new_tokens=40,
    use_cache=True,
    temperature=0.1,                 # Keep it focused, no creative rambling
    do_sample=False,                 # Greedy decoding for strict JSON
    eos_token_id=tokenizer.eos_token_id, # Force it to respect the stop token
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- THE MODEL'S WORK ---")
print(response)